# Pre- and Postsynaptic Components

Generates Fig. 1 (`pre-post-components.pdf`). This notebook reconstructs the weak-coupling factorization of the Fisher-information gradient into postsynaptic and presynaptic components for a representative sequential input. It illustrates how short-term depression makes the presynaptic term onset-sensitive and why the temporal overlap is larger for anti-causal pairings.


In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy import signal
from numba import jit

from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURE_DIR = ROOT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Shared plotting style for manuscript PDFs.
config = {
    "font.family": "sans-serif",
    "font.size": 18.0,
    "axes.titlelocation": "left",
    "axes.titlesize": 19.0,
    "axes.labelsize": 19.0,
    "xtick.labelsize": 17.0,
    "ytick.labelsize": 17.0,
    "legend.fontsize": 17.0,
    "figure.titlesize": 19.0,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.5,
    "lines.markersize": 4.0,
    "patch.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "axes.xmargin": 0.01,
    "axes.ymargin": 0.05,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.minor.size": 2.0,
    "ytick.minor.size": 2.0,
    "xtick.minor.width": 0.6,
    "ytick.minor.width": 0.6,
    "legend.frameon": False,
    "legend.fancybox": False,
    "image.interpolation": "none",
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}
plt.rcParams.update(config)


In [ ]:
# Define activation functions
def activation_function_exp(u, params):
    """Exponential firing-rate nonlinearity."""
    return params['g_M'] * np.exp(params['beta'] * (u - params['u_c']))

def activation_function_sigmoid(u, params):
    """Sigmoid firing-rate nonlinearity."""
    from scipy.special import expit
    return params['g_M'] * expit(params['beta'] * (u - params['u_c']))

rectified_cos = lambda theta , theta_c: np.maximum(0, (np.cos(theta) - np.cos(theta_c)))

@jit(nopython=True)
def convolve_epsp(signal_in, tau_s, dt):
    """ Convolve input signal with EPSP kernel using recursive filter. """
    a = np.exp(-dt/tau_s)
    b = tau_s * (1 - a)
    signal_out = np.zeros_like(signal_in)
    for i in range(1, len(signal_in)):
        signal_out[i] = a * signal_out[i-1] + b * signal_in[i-1]
    return signal_out

# compute pre factor
def compute_pre_factor(h_pre, t_eval, U, w0, params, activation='exp'):
    """
    Compute the pre-synaptic factor for gradient calculation with STP.
        h_pre: pre-synaptic (external) input (func)
        t_eval: time points for evaluation (array)
        U: release prob.
        w0: post-synaptic weight
        activation: type of activation function ('exp' or 'sigmoid')
    """
    if activation == 'exp':
        activation_function = lambda u: activation_function_exp(u, params=params)
    elif activation == 'sigmoid':
        activation_function = lambda u: activation_function_sigmoid(u, params=params)
    else:
        raise ValueError("Unsupported activation function")
    def dynamics_STP_factor(t, y, U, nu0_func, tau_d):
        '''
        dynamics of STP factor.
            w: synaptic efficacy
            f_w0: derivative with respect to w0
            f_U: derivative with respect to U
            U: release prob.
            nu0_func: pre firing rate (as func of time)
            tau_d: STD time constant
        '''
        f_w0, f_U = y
        nu0_t = nu0_func(t)

        recovery = 1 / tau_d
        usage = nu0_t * U

        df_w0_dt = recovery * (1-f_w0) - usage * f_w0
        df_U_dt = recovery * (1 - f_U) - usage * (f_U + f_w0)
        return [df_w0_dt, df_U_dt]

    # unpack parameters
    tau_d = params['tau_d']
    tau_s = params['tau_s'] # EPSP time constant (s)
    t = t_eval
    t0 = t[0]
    t1 = t[-1]
    dt = t[1] - t[0]

    nu0_pre = activation_function(h_pre(t)) # array (Nt,) : pre firing rate

    # compute STP factor
    nu0_func = lambda t_val: activation_function(h_pre(t_val))
    f0 = [1/(1 + tau_d * nu0_pre[0] * U),1/(1 + tau_d * nu0_pre[0] * U)**2] # init values for f = [f_w0, f_U]


    # define max_step
    if 'freq' in params and params['freq'] and params['freq'] > 0:
        max_step = 1 / (10 * params['freq'])
    else:
        max_step = dt * 10

    sol_f = solve_ivp(fun=lambda t, y: dynamics_STP_factor(t, y, U, nu0_func, tau_d),
        t_span = (t0, t1),
        y0=f0,
        t_eval=t,
        method='BDF',
        max_step=max_step
    )

    f_w0 = sol_f.y[0,:] # array (Nt,) : f_w0 at each time point
    f_U = sol_f.y[1,:] # array (Nt,) : f_U at each time point
    pre_factor_w0 = nu0_pre * f_w0 * U
    pre_factor_U = nu0_pre * f_U * w0
    pre_factor_w_noSTP = nu0_pre

    return {
        'pre_factor_w0': pre_factor_w0,
        'pre_factor_U': pre_factor_U,
        'pre_factor_w_noSTP': pre_factor_w_noSTP,
        'nu0_pre': nu0_pre,
        'h_pre': h_pre(t),
        'f_w0': f_w0,
        'f_U': f_U
    }
# compute post factor
def compute_post_factor(h_post, h_prime, t_eval, params, activation='exp'):
    """
    Compute the post-synaptic factor for gradient calculation.
        h_post: post-synaptic (external) input (func)
        h_prime: derivative of h_post (with respect to encoding param) (func)
        t_eval: time points for evaluation (array)
        activation: type of activation function ('exp' or 'sigmoid')
    """

    if activation == 'exp':
        activation_function = lambda u: activation_function_exp(u, params=params)
    elif activation == 'sigmoid':
        activation_function = lambda u: activation_function_sigmoid(u, params=params)
    else:
        raise ValueError("Unsupported activation function")

    t = t_eval
    t0 = t[0]
    t1 = t[-1]
    dt = t[1] - t[0]

    nu0_post = activation_function(h_post(t)) # array (Nt,) : post firing rate

    if activation == 'exp':
        eta = (params['beta'] ** 3) * (h_prime(t) ** 2)
    elif activation == 'sigmoid':
        eta = (params['beta'] ** 3) * (h_prime(t) ** 2) * ((1 - nu0_post / params['g_M']) ** 2) * (1 - 3 * nu0_post / params['g_M'])
    post_factor = nu0_post * eta
    return {'post_factor': post_factor, 'nu0_post': nu0_post, 'h_post': h_post(t), 'h_prime': h_prime(t), 'eta': eta}

# compute gradient
def compute_gradient(pre_factor, post_factor, t_eval, params, t_span_grad=None):
    """
    Compute the gradient of the synaptic weight with respect to encoding parameters.
        pre_factor: pre-synaptic factor (array)
        post_factor: post-synaptic factor (array)
        t_eval: time points for evaluation (array)
        tau_s: EPSP time constant (s)
    """
    tau_s = params['tau_s']
    dt = t_eval[1] - t_eval[0]

    # Convolve pre_factor with EPSP kernel
    convolved_pre = convolve_epsp(pre_factor, tau_s, dt)
    pre_post_product = convolved_pre * post_factor

    if t_span_grad is not None:
        # Restrict to specified time span for gradient calculation
        mask = (t_eval >= t_span_grad[0]) & (t_eval <= t_span_grad[1])
    else:
        mask = np.ones_like(t_eval, dtype=bool)
    # Compute gradient as integral of convolved pre and post factors
    gradient = np.trapezoid(pre_post_product[mask], t_eval[mask])

    return { 'gradient': gradient, 'pre_post_product': pre_post_product }
def compute_gradient_all(result_pre, result_post, t_eval, params, t_span_grad=None):
    """ Compute gradients for all pre factors. """
    gradients = {}
    pre_post_products = {}
    for key in ['pre_factor_w0', 'pre_factor_U', 'pre_factor_w_noSTP']:
        res = compute_gradient(result_pre[key], result_post['post_factor'], t_eval, params, t_span_grad)
        output_key = key.replace('pre_factor_', '')
        gradients[output_key] = res['gradient']
        pre_post_products[output_key] = res['pre_post_product']
    return gradients, pre_post_products


In [ ]:

def plot_pre_post_factors(t_eval, result_pre, result_post_causal, result_post_anti, params, t_span_grad = None, t_span_plot=None, figsize=(12, 14)):
    """ Plot pre and post factors. """
    if t_span_plot is not None:
        t0, t1 = t_span_plot
    else:
        t0, t1 = t_eval[0], t_eval[-1]

    mask = (t_eval >= t0) & (t_eval <= t1)
    t_plot = t_eval[mask]
    fig, axes = plt.subplots(6, 1, figsize=figsize, sharex=True)

    # 1. h_pre
    ax = axes[0]
    ax.plot(t_plot, result_pre['h_pre'][mask], color='blue')
    ax.set_ylabel('h(t)')
    ax.grid(True, alpha=0.3)
    ax.set_title(r'External input $h_{pre}(t)$')

    # 2. nu0_pre
    ax = axes[1]
    ax.plot(t_plot, result_pre['nu0_pre'][mask], color='blue')
    ax.set_ylabel(r'$\nu^0(t) [Hz]$')
    ax.grid(True, alpha=0.3)
    ax.set_title(r'Firing rate $\nu^0_{pre}(t)$')

    # 3. STP_factor (f_w0, f_U)
    ax = axes[2]
    ax.plot(t_plot, result_pre['f_w0'][mask], label=r'$w_0$', color='green')
    ax.plot(t_plot, result_pre['f_U'][mask], label=r'$U$', color='orange')
    ax.set_ylabel(r'$f^Z (t)$')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)
    ax.set_title(r'STP factors $f^Z (t)$')

    # 4. pre_factor (w0, U, noSTP)
    def normalize_signal(signal):
        """"""
        norm = np.max(np.abs(signal))
        if norm > 0:
            return signal / norm
        else:
            return signal
    def normalize_pair(signal_a, signal_b):
        norm = max(np.max(np.abs(signal_a)), np.max(np.abs(signal_b)))
        if norm > 0:
            return signal_a / norm, signal_b / norm
        else:
            return signal_a, signal_b
    def set_relative_axis(ax, y_min, y_max, padding=0.05):
        y_min = min(y_min, 0.0)
        y_max = max(y_max, 0.0)
        if y_min == y_max:
            pad = 1.0 if y_min == 0 else abs(y_min) * 0.1
            y_min -= pad
            y_max += pad
        else:
            pad = (y_max - y_min) * padding
            y_min -= pad
            y_max += pad
        ax.set_ylim(y_min, y_max)
        ax.set_yticks([0])
    pre_factor_w0 = normalize_signal(result_pre['pre_factor_w0'][mask])
    pre_factor_U = normalize_signal(result_pre['pre_factor_U'][mask])
    pre_factor_w_noSTP = normalize_signal(result_pre['pre_factor_w_noSTP'][mask])

    ax = axes[3]
    ax.plot(t_plot, pre_factor_w0,
            label=r'$w_0$', color='green')
    ax.plot(t_plot, pre_factor_U,
            label=r'$U$', color='orange')
    ax.plot(t_plot, pre_factor_w_noSTP,
            label=r'$w_{\mathrm{static}}$', color='gray')
    ax.set_ylabel(r'$C_{pre}^Z (t)$')
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)
    ax.grid(True, alpha=0.3)
    ax.set_title(r'Pre component $C_{pre}^Z (t)$')
    pre_min = min(pre_factor_w0.min(), pre_factor_U.min(), pre_factor_w_noSTP.min())
    pre_max = max(pre_factor_w0.max(), pre_factor_U.max(), pre_factor_w_noSTP.max())
    set_relative_axis(ax, pre_min, pre_max)

    # 5. post_factor
    ax = axes[4]
    post_causal = result_post_causal['post_factor'][mask]
    post_anti = result_post_anti['post_factor'][mask]
    ax.plot(t_plot, post_anti, color='red', linestyle='-', label='anti-causal')
    ax.plot(t_plot, post_causal, color='red', linestyle='--', label='causal')
    ax.set_ylabel(r'$C_{post} (t)$')
    ax.grid(True, alpha=0.3)
    ax.set_title(r'Post component $C_{post} (t)$')
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)
    post_min = min(post_causal.min(), post_anti.min())
    post_max = max(post_causal.max(), post_anti.max())
    set_relative_axis(ax, post_min, post_max)

    # 6. overlap (convolved pre * post)
    ax = axes[5]
    gradients_causal, pre_post_products_causal = compute_gradient_all(result_pre, result_post_causal, t_eval, params, t_span_grad)
    gradients_anti, pre_post_products_anti = compute_gradient_all(result_pre, result_post_anti, t_eval, params, t_span_grad)
    overlap_w0_causal = pre_post_products_causal['w0'][mask]
    overlap_w0_anti = pre_post_products_anti['w0'][mask]
    overlap_w0_causal, overlap_w0_anti = normalize_pair(overlap_w0_causal, overlap_w0_anti)
    overlap_U_causal = pre_post_products_causal['U'][mask]
    overlap_U_anti = pre_post_products_anti['U'][mask]
    overlap_U_causal, overlap_U_anti = normalize_pair(overlap_U_causal, overlap_U_anti)
    overlap_noSTP_causal = pre_post_products_causal['w_noSTP'][mask]
    overlap_noSTP_anti = pre_post_products_anti['w_noSTP'][mask]
    overlap_noSTP_causal, overlap_noSTP_anti = normalize_pair(overlap_noSTP_causal, overlap_noSTP_anti)
    ax.plot(t_plot, overlap_w0_causal, color='green', linestyle='--')
    ax.plot(t_plot, overlap_w0_anti, color='green', label=r'$w_0$', linestyle='-')
    ax.plot(t_plot, overlap_U_causal,  color='orange', linestyle='--')
    ax.plot(t_plot, overlap_U_anti, color='orange', label=r'$U$',  linestyle='-')
    ax.plot(t_plot, overlap_noSTP_causal, color='gray', linestyle='--')
    ax.plot(t_plot, overlap_noSTP_anti, color='gray', label=r'$w_{\mathrm{static}}$',  linestyle='-')

    ax.set_ylabel(r'$\delta \dot{J} (t) / \delta Z$')
    ax.set_xlabel('Time [s]')
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)
    ax.grid(True, alpha=0.3)
    min_y = min(
        overlap_w0_causal.min(), overlap_w0_anti.min(),
        overlap_U_causal.min(), overlap_U_anti.min(),
        overlap_noSTP_causal.min(), overlap_noSTP_anti.min()
    )
    max_y = max(
        overlap_w0_causal.max(), overlap_w0_anti.max(),
        overlap_U_causal.max(), overlap_U_anti.max(),
        overlap_noSTP_causal.max(), overlap_noSTP_anti.max()
    )
    set_relative_axis(ax, min_y, max_y)
    ax.set_title("Instantaneous gradient")

    plt.tight_layout()
    print(params)
    return fig, axes


In [ ]:
# Parameter set used for the published figure.
params = {
    'freq': 1.0,
    'amp': 2.0,
    'tau_d': 0.5,  # Depression time constant (s)
    'beta' : 3.0,  # Steepness of activation function
    'g_M' : 10.0, # Maximum firing rate (Hz) for sigmoid activation.
    'u_c' : 1.0,    # Activation threshold
    'tau_s' : 0.01  # Synaptic time constant (s)
}

h_rect = lambda z, t: params['amp'] * rectified_cos(2 * np.pi * params['freq'] * t - z, theta_c=np.pi/2)
h_pre = lambda t: h_rect(np.pi, t)
delta_z = np.pi/4
h_post_causal = lambda t: h_rect(np.pi + delta_z, t)
h_prime_causal = lambda t: np.where(h_post_causal(t) > 0, -params['amp'], 0)

h_post_anti = lambda t: h_rect(np.pi - delta_z, t)
h_prime_anti = lambda t: np.where(h_post_anti(t) > 0, -params['amp'], 0)

t0 = -2.5/params['freq']; t1 = 1.0 / params['freq']
dt = 0.001
t_eval = np.arange(t0, t1, dt)
t_span_grad = (0, t1)

result_pre = compute_pre_factor(h_pre, t_eval, U=0.15, w0=1.0, params=params, activation='exp')
result_post_causal = compute_post_factor(h_post_causal, h_prime_causal, t_eval, params=params, activation='exp')
result_post_anti = compute_post_factor(h_post_anti, h_prime_anti, t_eval, params=params, activation='exp')

fig, axes = plot_pre_post_factors(t_eval, result_pre, result_post_causal, result_post_anti, params, t_span_grad=t_span_grad, t_span_plot = t_span_grad, figsize=(8, 12))
plt.savefig(FIGURE_DIR / 'pre-post-components.pdf', bbox_inches='tight')
plt.show()
